In [1]:
import polars as pl

In [2]:
#icu
icu = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260902_083216-roberta_17526270_y_icu_readmit_30_overlap_256_with_proto_5/files/ckpt/test_predictions.csv'

# ymort
mort = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260907_165450-roberta_17786113_y_mort_overlap_256_with_proto_17/files/ckpt/test_predictions.csv'

# los
los = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260910_130525-roberta_17843963_y_los_7_visit_256_with_proto_14/files/ckpt/test_predictions.csv'

# 12mo
mort12 = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260909_003911-roberta_17788223_y_mort_12mo_care_stage_256_with_proto_14/files/ckpt/test_predictions.csv'


In [3]:
a = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260902_083216-roberta_17526270_y_icu_readmit_30_overlap_256_with_proto_5/files/ckpt/test_predictions.csv'

In [4]:
b = '/scratch/sas10092/ehr-foundation/models/final/wandb/run-20260830_154819-roberta_base_17505328_y_icu_readmit_30_baselines_final/files/ckpt/test_predictions.csv'

In [5]:
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.notebook import tqdm

def significance_stars(p_value: float) -> str:
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def paired_bootstrap_significance(
    model_path: str,
    baseline_path: str,
    num_bootstrap: int = 1000,
    seed: int = 42,
) -> dict:
    keys = ["subject_id", "hadm_id", "icustay_id"]

    model = (
        pl.read_csv(model_path)
        .select(keys + ["label", "prediction"])
        .rename({
            "label": "model_label",
            "prediction": "model_prediction",
        })
    )

    baseline = (
        pl.read_csv(baseline_path)
        .select(keys + ["label", "prediction"])
        .rename({
            "label": "baseline_label",
            "prediction": "baseline_prediction",
        })
    )

    paired = model.join(
        baseline,
        on=keys,
        how="inner",
        validate="1:1",
    )

    if paired.height == 0:
        raise ValueError("No matching examples were found.")

    if paired.height != model.height or paired.height != baseline.height:
        raise ValueError(
            "The files do not contain exactly the same examples. "
            f"Matched={paired.height}, model={model.height}, "
            f"baseline={baseline.height}."
        )

    mismatches = paired.filter(
        pl.col("model_label") != pl.col("baseline_label")
    ).height

    if mismatches:
        raise ValueError(
            f"Labels differ for {mismatches} matched examples."
        )

    paired = paired.sort(keys)

    y_true = paired["model_label"].to_numpy()
    model_score = paired["model_prediction"].to_numpy()
    baseline_score = paired["baseline_prediction"].to_numpy()
    subject_ids = paired["subject_id"].to_numpy()

    if not (
        np.isfinite(y_true).all()
        and np.isfinite(model_score).all()
        and np.isfinite(baseline_score).all()
    ):
        raise ValueError("The input contains missing or non-finite values.")

    if np.unique(y_true).size != 2:
        raise ValueError("Both outcome classes are required.")

    # Map every patient to all corresponding rows.
    unique_subjects = np.unique(subject_ids)
    subject_rows = {
        subject: np.flatnonzero(subject_ids == subject)
        for subject in unique_subjects
    }

    # Point estimates and observed differences.
    model_auroc = roc_auc_score(y_true, model_score)
    baseline_auroc = roc_auc_score(y_true, baseline_score)

    model_auprc = average_precision_score(y_true, model_score)
    baseline_auprc = average_precision_score(y_true, baseline_score)

    observed_auroc_difference = model_auroc - baseline_auroc
    observed_auprc_difference = model_auprc - baseline_auprc

    rng = np.random.default_rng(seed)

    bootstrap_auroc_differences = []
    bootstrap_auprc_differences = []

    for _ in tqdm(range(num_bootstrap)):
        sampled_subjects = rng.choice(
            unique_subjects,
            size=len(unique_subjects),
            replace=True,
        )

        # Concatenation preserves multiplicity when a patient is sampled
        # more than once.
        sampled_indices = np.concatenate(
            [subject_rows[subject] for subject in sampled_subjects]
        )

        y_sample = y_true[sampled_indices]

        # AUROC is undefined when a resample contains only one class.
        if np.unique(y_sample).size != 2:
            continue

        model_sample = model_score[sampled_indices]
        baseline_sample = baseline_score[sampled_indices]

        bootstrap_auroc_differences.append(
            roc_auc_score(y_sample, model_sample)
            - roc_auc_score(y_sample, baseline_sample)
        )

        bootstrap_auprc_differences.append(
            average_precision_score(y_sample, model_sample)
            - average_precision_score(y_sample, baseline_sample)
        )

    auroc_differences = np.asarray(bootstrap_auroc_differences)
    auprc_differences = np.asarray(bootstrap_auprc_differences)

    if len(auroc_differences) == 0:
        raise RuntimeError("All bootstrap samples contained only one class.")

    # Center the bootstrap distributions to represent the null hypothesis.
    centered_auroc = auroc_differences - observed_auroc_difference
    centered_auprc = auprc_differences - observed_auprc_difference

    # Two-sided bootstrap p-values with finite-sample correction.
    auroc_p = (
        np.sum(
            np.abs(centered_auroc)
            >= abs(observed_auroc_difference)
        )
        + 1
    ) / (len(centered_auroc) + 1)

    auprc_p = (
        np.sum(
            np.abs(centered_auprc)
            >= abs(observed_auprc_difference)
        )
        + 1
    ) / (len(centered_auprc) + 1)

    return {
        "n_patients": len(unique_subjects),
        "valid_bootstrap_samples": len(auroc_differences),

        "model_auroc": model_auroc,
        "baseline_auroc": baseline_auroc,
        "auroc_difference": observed_auroc_difference,
        "auroc_p_value": float(auroc_p),
        "auroc_significance": significance_stars(auroc_p),

        "model_auprc": model_auprc,
        "baseline_auprc": baseline_auprc,
        "auprc_difference": observed_auprc_difference,
        "auprc_p_value": float(auprc_p),
        "auprc_significance": significance_stars(auprc_p),
    }

In [6]:
# result = paired_bootstrap_significance(
#     model_path=a,
#     baseline_path=b,
#     num_bootstrap=10_000,
#     seed=42,
# )

# print(
#     f"AUROC: Δ={result['auroc_difference']:.3f}, "
#     f"p={result['auroc_p_value']:.4g} "
#     f"{result['auroc_significance']}"
# )

# print(
#     f"AUPRC: Δ={result['auprc_difference']:.3f}, "
#     f"p={result['auprc_p_value']:.4g} "
#     f"{result['auprc_significance']}"
# )

In [7]:
from pathlib import Path

root = Path("/scratch/sas10092/ehr-foundation/models/final/wandb")

csv_paths = [
    str(csv_path)
    for run_dir in root.glob("run-*")
    if run_dir.is_dir()
    for csv_path in run_dir.rglob("*.csv")
]



In [35]:
# from pathlib import Path


# baseline_root = Path(
#     "/scratch/sas10092/ehr-foundation/models/final/wandb"
# )

# model_paths = {
#     "y_icu_readmit_30": icu,
#     "y_mort": mort,
#     "y_los_7": los,
#     "y_mort_12mo": mort12,
# }

# task_order = [
#     "y_icu_readmit_30",
#     "y_mort",
#     "y_los_7",
#     "y_mort_12mo",
# ]


# def parse_baseline_path(path):
#     run_name = path.parents[2].name

#     # Remove: run-YYYYMMDD_HHMMSS-
#     run_info = run_name.split("-", maxsplit=2)[2]

#     for task in task_order:
#         task_suffix = f"_{task}_baselines_final"

#         if run_info.endswith(task_suffix):
#             model_and_job = run_info.removesuffix(task_suffix)

#             # Remove the final job-ID component.
#             model_name = model_and_job.rsplit("_", maxsplit=1)[0]

#             return model_name, task

#     raise ValueError(f"Could not parse path: {path}")


# baseline_files = sorted(
#     baseline_root.glob("run-*/files/ckpt/test_predictions.csv")
# )

# parsed_files = []

# for baseline_path in baseline_files:
#     baseline_name, task = parse_baseline_path(baseline_path)

#     parsed_files.append({
#         "baseline": baseline_name,
#         "task": task,
#         "path": str(baseline_path),
#     })

# parsed_files.sort(
#     key=lambda item: (
#         item["baseline"],
#         task_order.index(item["task"]),
#     )
# )


# for item in parsed_files:
#     task = item["task"]
#     print(
#         f"{item['baseline']:25s} \n\n"
#         f"{task:20s}\n\n"
#         f"Baseline: {item['path']}\n\n"
#         f"Model:    {model_paths[task]}\n\n"
#     )
#     baseline_name = item["baseline"]
    

#     result = paired_bootstrap_significance(
#         model_path=model_paths[task],
#         baseline_path=item["path"],
#         num_bootstrap=10_000,
#         seed=42,
#     )

#     print(baseline_name)
#     print(task)

#     print(
#         f"AUROC: {result['auroc_significance']} "
#         f"(p={result['auroc_p_value']:.4g})"
#     )

#     print(
#         f"AUPRC: {result['auprc_significance']} "
#         f"(p={result['auprc_p_value']:.4g})"
#     )

#     print()

In [36]:
from pathlib import Path
import polars as pl


without_root = Path(
    "/scratch/sas10092/ehr-foundation/models/final/wandb"
)

with_root = Path(
    "/scratch/sas10092/ehr-foundation/models/1_last"
)

variants = {
    "bert_medbert": "Med-BERT",
    "bert_cehrbert": "CEHR-BERT",
    "bert_behrt": "BEHRT",
    "bert_hibehrt": "Hi-BEHRT",
}

# Longest names must come first so y_mort is not confused
# with y_mort_12mo.
tasks = [
    "y_icu_readmit_30",
    "y_mort_12mo",
    "y_los_7",
    "y_mort",
]

display_task_order = [
    "y_icu_readmit_30",
    "y_mort",
    "y_los_7",
    "y_mort_12mo",
]


def identify_model_and_task(path: Path):
    run_name = path.parents[2].name

    matched_models = [
        model
        for model in variants
        if model in run_name
    ]

    matched_tasks = [
        task
        for task in tasks
        if f"_{task}_" in run_name
    ]

    if len(matched_models) != 1:
        return None

    # Keep the longest task match. This resolves y_mort_12mo
    # versus y_mort.
    if not matched_tasks:
        return None

    task = max(matched_tasks, key=len)

    return matched_models[0], task


def discover_predictions(root: Path):
    discovered = {}

    paths = sorted(
        root.glob("run-*/files/ckpt/test_predictions.csv")
    )

    for path in paths:
        identity = identify_model_and_task(path)

        if identity is None:
            continue

        if identity in discovered:
            previous = discovered[identity]

            raise ValueError(
                "Multiple prediction files found for "
                f"{identity}:\n"
                f"  1. {previous}\n"
                f"  2. {path}\n"
                "Select the intended run before continuing."
            )

        discovered[identity] = path

    return discovered


without_predictions = discover_predictions(without_root)
with_predictions = discover_predictions(with_root)

expected_pairs = {
    (model, task)
    for model in variants
    for task in tasks
}

missing_without = expected_pairs - set(without_predictions)
missing_with = expected_pairs - set(with_predictions)

if missing_without:
    raise ValueError(
        "Missing files without EHR-RAGp:\n"
        + "\n".join(
            f"  {model} | {task}"
            for model, task in sorted(missing_without)
        )
    )

if missing_with:
    raise ValueError(
        "Missing files with EHR-RAGp:\n"
        + "\n".join(
            f"  {model} | {task}"
            for model, task in sorted(missing_with)
        )
    )

print("File validation passed: 4 models × 4 tasks.")
print()

# Display every matched pair before starting the bootstrap.
for model in variants:
    for task in display_task_order:
        identity = (model, task)

        print(f"{variants[model]} | {task}")
        print(f"Without: {without_predictions[identity]}")
        print(f"With:    {with_predictions[identity]}")
        print()


results = []

for model in variants:
    for task in display_task_order:
        identity = (model, task)

        result = paired_bootstrap_significance(
            model_path=str(with_predictions[identity]),
            baseline_path=str(without_predictions[identity]),
            num_bootstrap=10_000,
            seed=42,
        )

        results.append({
            "model": variants[model],
            "task": task,
            "auroc_difference": result["auroc_difference"],
            "auroc_p_value": result["auroc_p_value"],
            "auroc_significance": result["auroc_significance"],
            "auprc_difference": result["auprc_difference"],
            "auprc_p_value": result["auprc_p_value"],
            "auprc_significance": result["auprc_significance"],
        })

        print(variants[model])
        print(task)
        print(
            f"AUROC: {result['auroc_significance']} "
            f"(p={result['auroc_p_value']:.4g})"
        )
        print(
            f"AUPRC: {result['auprc_significance']} "
            f"(p={result['auprc_p_value']:.4g})"
        )
        print()


results_df = pl.DataFrame(results)

results_df.write_csv(
    "ehr_ragp_variant_significance.csv"
)

results_df

File validation passed: 4 models × 4 tasks.

Med-BERT | y_icu_readmit_30
Without: /scratch/sas10092/ehr-foundation/models/final/wandb/run-20260831_101707-bert_medbert_17505328_y_icu_readmit_30_baselines_final/files/ckpt/test_predictions.csv
With:    /scratch/sas10092/ehr-foundation/models/1_last/run-20260913_051821-bert_medbert_17914452_y_icu_readmit_30_overlap_256_with_proto_22/files/ckpt/test_predictions.csv

Med-BERT | y_mort
Without: /scratch/sas10092/ehr-foundation/models/final/wandb/run-20260831_193953-bert_medbert_17505331_y_mort_baselines_final/files/ckpt/test_predictions.csv
With:    /scratch/sas10092/ehr-foundation/models/1_last/run-20260913_200610-bert_medbert_17914453_y_mort_overlap_256_with_proto_22/files/ckpt/test_predictions.csv

Med-BERT | y_los_7
Without: /scratch/sas10092/ehr-foundation/models/final/wandb/run-20260831_165238-bert_medbert_17505332_y_los_7_baselines_final/files/ckpt/test_predictions.csv
With:    /scratch/sas10092/ehr-foundation/models/1_last/run-2026091

  0%|          | 0/10000 [00:00<?, ?it/s]

Med-BERT
y_icu_readmit_30
AUROC: *** (p=0.0002)
AUPRC: ** (p=0.004)



  0%|          | 0/10000 [00:00<?, ?it/s]

Med-BERT
y_mort
AUROC: *** (p=9.999e-05)
AUPRC: *** (p=9.999e-05)



...

  0%|          | 0/10000 [00:00<?, ?it/s]

Hi-BEHRT
y_los_7
AUROC: *** (p=9.999e-05)
AUPRC: *** (p=9.999e-05)



  0%|          | 0/10000 [00:00<?, ?it/s]

Hi-BEHRT
y_mort_12mo
AUROC: *** (p=9.999e-05)
AUPRC: *** (p=9.999e-05)



model,task,auroc_difference,auroc_p_value,auroc_significance,auprc_difference,auprc_p_value,auprc_significance
str,str,f64,f64,str,f64,f64,str
"""Med-BERT""","""y_icu_readmit_30""",0.051094,0.0002,"""***""",0.035195,0.004,"""**"""
"""Med-BERT""","""y_mort""",0.04511,0.0001,"""***""",0.123097,0.0001,"""***"""
"""Med-BERT""","""y_los_7""",0.03009,0.0001,"""***""",0.042872,0.0004,"""***"""
"""Med-BERT""","""y_mort_12mo""",0.09421,0.0001,"""***""",0.137126,0.0001,"""***"""
"""CEHR-BERT""","""y_icu_readmit_30""",0.09453,0.0001,"""***""",0.04123,0.006799,"""**"""
…,…,…,…,…,…,…,…
"""BEHRT""","""y_mort_12mo""",0.058273,0.0001,"""***""",0.107482,0.0001,"""***"""
"""Hi-BEHRT""","""y_icu_readmit_30""",0.063467,0.0008,"""***""",0.03096,0.017898,"""*"""
"""Hi-BEHRT""","""y_mort""",0.047614,0.0001,"""***""",0.148228,0.0001,"""***"""


In [19]:
a = '/scratch/sas10092/ehr-foundation/models/1_last/run-20260907_165450-roberta_17786113_y_mort_overlap_256_with_proto_17/files/ckpt/test_predictions.csv'
b= '/scratch/sas10092/ehr-foundation/models/ehr-ragp/wandb/run-20260923_220220-roberta_18066036_y_mort_overlap_256_without_proto_5/files/ckpt/test_predictions.csv'
result = paired_bootstrap_significance(
    model_path=a,
    baseline_path=b,
    num_bootstrap=10_000,
    seed=42,
)

print(
    f"AUROC: Δ={result['auroc_difference']:.3f}, "
    f"p={result['auroc_p_value']:.4g} "
    f"{result['auroc_significance']}"
)

print(
    f"AUPRC: Δ={result['auprc_difference']:.3f}, "
    f"p={result['auprc_p_value']:.4g} "
    f"{result['auprc_significance']}"
)


  0%|          | 0/10000 [00:00<?, ?it/s]

AUROC: Δ=0.008, p=0.0002 ***
AUPRC: Δ=0.023, p=0.006599 **
